In [30]:
import os
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from IPython.display import HTML
import joblib

if torch.backends.mps.is_available():
    device = torch.device("mps")      # Mac GPU (Apple Silicon)
elif torch.cuda.is_available():
    device = torch.device("cuda")     # Nvidia GPU
else:
    device = torch.device("cpu")

In [31]:
import numpy as np 
def select_equally_spaced_frames(data, num_frames=30, axis=0):
    """
    Select equally spaced frames from n-dimensional data along specified axis.
    
    Parameters:
    -----------
    data : numpy array or list
        Input data (2D, 4D, or any dimension)
    num_frames : int
        Number of frames to select (default: 30)
    axis : int
        Axis along which to select frames (default: 0, typically the time/frame dimension)
    
    Returns:
    --------
    numpy array
        Selected frames with same dimensions as input except the selected axis
    """
    # Convert to numpy array if it's a list
    if not isinstance(data, np.ndarray):
        data = np.array(data)
    
    # Get total number of frames along the specified axis
    total_frames = data.shape[axis]
    
    # Generate equally spaced indices
    indices = np.linspace(0, total_frames - 1, num_frames).astype(int)

    
    # Select using advanced indexing
    # Build a tuple of slices for indexing
    selector = [slice(None)] * data.ndim
    selector[axis] = indices
    
    return data[tuple(selector)]



In [32]:
def convert_padded_csv_to_fixed_c(
    input_folder,
    output_folder,
    C=30):
    
    os.makedirs(output_folder, exist_ok=True)

    # ------------------------------------------
    # Trim each video based on previus assignemnt
    # ------------------------------------------
    for file_name in os.listdir(input_folder):

        if not file_name.endswith(".csv"):
            continue

        print(f"\nProcessing: {file_name}")

        csv_path = os.path.join(input_folder, file_name)

        df = pd.read_csv(csv_path)

        target = df["target"].iloc[0]

        X_flat = df.drop(columns=["target"]).values

        X = X_flat.reshape(-1, 39)

        non_zero_mask = ~(np.all(X == 0, axis=1))
        X_trimmed = X[non_zero_mask]

        print(f"Original frames: {len(X)}")
        print(f"After zero removal: {len(X_trimmed)}")


        # -----------------------------------------
        # Convert to fixed C frames
        # -----------------------------------------

        if len(X_trimmed) < C:
            print(f"Skipping {file_name}: too short")
            continue

        X_fixed = select_equally_spaced_frames(X_trimmed, num_frames=C, axis=0)

        
        # -----------------------------------------
        # Flatten + save
        # -----------------------------------------
        row = X_fixed.flatten().tolist()
        row.append(target)

        columns = []

        joints = [
            "head",
            "left_shoulder", "left_elbow",
            "right_shoulder", "right_elbow",
            "left_hand", "right_hand",
            "left_hip", "right_hip",
            "left_knee", "right_knee",
            "left_foot", "right_foot"
        ]

        for frame_idx in range(C):
            for joint in joints:
                columns += [
                    f"frame{frame_idx}_{joint}_x",
                    f"frame{frame_idx}_{joint}_y",
                    f"frame{frame_idx}_{joint}_z"
                ]

        columns.append("target")

        output_df = pd.DataFrame([row], columns=columns)

        output_path = os.path.join(
            output_folder,
            file_name.replace("_padded", "_fixedC")
        )

        output_df.to_csv(output_path, index=False)

        print(f"Saved: {output_path}")

    print("\nDone.")

In [ ]:
convert_padded_csv_to_fixed_c(
    input_folder="../../MainProject/data/mediapipe_padded_videos",
    output_folder="../../MainProject/data/mediapipe_bad_good_fixed_c",
    C=30
)


Processing: G22_padded.csv
Original frames: 173
After zero removal: 66
Saved: ../../MainProject/data/mediapipe__bad_good_fixed_c/G22_fixedC.csv

Processing: G40_padded.csv
Original frames: 173
After zero removal: 66
Saved: ../../MainProject/data/mediapipe__bad_good_fixed_c/G40_fixedC.csv

Processing: G77_padded.csv
Original frames: 173
After zero removal: 63
Saved: ../../MainProject/data/mediapipe__bad_good_fixed_c/G77_fixedC.csv

Processing: G09_padded.csv
Original frames: 173
After zero removal: 79
Saved: ../../MainProject/data/mediapipe__bad_good_fixed_c/G09_fixedC.csv

Processing: G66_padded.csv
Original frames: 173
After zero removal: 74
Saved: ../../MainProject/data/mediapipe__bad_good_fixed_c/G66_fixedC.csv

Processing: G51_padded.csv
Original frames: 173
After zero removal: 76
Saved: ../../MainProject/data/mediapipe__bad_good_fixed_c/G51_fixedC.csv

Processing: G04_padded.csv
Original frames: 173
After zero removal: 65
Saved: ../../MainProject/data/mediapipe__bad_good_fixed_c/

In [34]:
df = pd.read_csv("../../MainProject/data/mediapipe__bad_good_fixed_c/A1_fixedC.csv")

y = df["target"].values

X_flat = df.drop(columns=["target"]).values

max_frames = 30 
n_features = 39

X = X_flat.reshape(-1, max_frames, n_features)

print(X.shape)

print(X)
print(y)

(1, 30, 39)
[[[ 0.51550901  0.2496651  -0.24653962 ...  0.47840539  0.92122191
    0.13640046]
  [ 0.5167681   0.25215581 -0.23707871 ...  0.47841737  0.92104799
    0.13885181]
  [ 0.51641935  0.25245732 -0.23231845 ...  0.47913221  0.92120498
    0.12562826]
  ...
  [ 0.51427466  0.25570107 -0.19791007 ...  0.4775244   0.92251325
    0.14171991]
  [ 0.51523548  0.25334084 -0.23080425 ...  0.47753194  0.92153627
    0.12568238]
  [ 0.51562971  0.25090641 -0.20876606 ...  0.47746381  0.91950154
    0.1115904 ]]]
[1]
